# Prop Trading Funnel Analytics — Разбор воронки проп-трейдинговой платформы

**EN:** This notebook is a lightweight review layer over the production pipeline. The core deliverable is the reproducible code in `src/`, the SQL in `sql/`, and the generated artifacts in `outputs/`.

**RU:** Ноутбук — это обзорный слой поверх основного пайплайна. Ключевые материалы: воспроизводимый код в `src/`, SQL-запросы в `sql/`, сгенерированные артефакты в `outputs/`.

Датасет синтетический, но внутренне консистентный: 18 000 регистраций, 7 этапов воронки (регистрация → KYC → покупка челленджа → фаза 1 → фаза 2 → финансирование → выплата).

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
OUTPUTS = ROOT / 'outputs'

overall_funnel = pd.read_csv(OUTPUTS / 'overall_funnel.csv')
channel_funnel = pd.read_csv(OUTPUTS / 'funnel_by_acquisition_channel.csv')
challenge_funnel = pd.read_csv(OUTPUTS / 'funnel_by_challenge_type.csv')
cohorts = pd.read_csv(OUTPUTS / 'cohort_analysis_by_registration_month.csv')
segments = pd.read_csv(OUTPUTS / 'trader_segment_comparison.csv')
summary_text = (OUTPUTS / 'summary.md').read_text()
display(Markdown(summary_text))

## 1. Обзор воронки / Funnel Overview

In [ ]:
display(overall_funnel)

display(
    channel_funnel[
        [
            'acquisition_channel',
            'registrations',
            'registration_to_purchase_pct',
            'purchase_to_funded_pct',
            'funded_to_payout_pct',
            'gross_profit_proxy_usd',
        ]
    ].sort_values('registrations', ascending=False)
)

display(
    challenge_funnel[
        [
            'challenge_type',
            'challenges',
            'funded_rate_pct',
            'payout_rate_pct',
            'gross_profit_proxy_usd',
            'payout_exposure_to_fee_ratio',
        ]
    ].sort_values('challenges', ascending=False)
)

display(
    segments[
        [
            'segment_name',
            'registration_to_purchase_pct',
            'challenge_to_funded_pct',
            'funded_to_payout_pct',
            'gross_profit_proxy_usd',
            'misleading_top_funnel_flag',
        ]
    ].sort_values('gross_profit_proxy_usd').head(10)
)


## 2. Ключевые таблицы / Key Tables

**RU:** Ниже — четыре ключевых разреза: общая воронка, разбивка по каналам привлечения, по типам челленджей и по сегментам трейдеров. Обратите внимание: `gross_profit_proxy_usd` = выручка от сборов − экспозиция выплат. Отрицательное значение означает, что выплаты превышают сборы в данном сегменте.

**EN:** Four core slices: overall funnel, by acquisition channel, by challenge type, and by trader segment. Note: `gross_profit_proxy_usd` = fee revenue − payout exposure. Negative = payouts exceed fees in that segment.

In [ ]:
cohorts

## 3. Когортный анализ / Cohort Analysis

**RU:** Когорты сформированы по месяцу регистрации. Каждая строка — это доля пользователей из данной когорты, достигших каждого этапа воронки. Если конверсия в funded резко падает для одной когорты — это сигнал: либо изменилось качество трафика, либо продукт, либо рыночные условия в тот период.

**EN:** Cohorts are formed by registration month. Each row shows what share of that cohort reached each funnel stage. A sharp drop in funded conversion for one cohort is a signal worth investigating: traffic quality, product change, or market conditions.

In [ ]:
for filename in [
    'funnel_stage_chart.png',
    'conversion_by_acquisition_channel.png',
    'conversion_by_challenge_type.png',
    'cohort_progression_heatmap.png',
    'payout_exposure_by_segment.png',
    'behavior_vs_funnel_outcomes.png',
]:
    display(Image(filename=str(OUTPUTS / filename), width=900))

## 4. Визуализации / Charts

**RU:** Шесть графиков отражают: общую воронку, конверсию по каналам и типам челленджей, прогрессию когорт, экспозицию выплат по сегментам и связь поведения трейдера с исходом. Размер точки на последнем графике — количество нарушений правил.

**EN:** Six charts covering: the overall funnel, conversion by channel and challenge type, cohort progression heatmap, payout exposure by segment, and trader behavior vs outcome. Bubble size on the last chart reflects rule violation count.

## 5. Выводы / Analyst Takeaways

### RU — Основные выводы

**Где воронка ломается.**
Крупнейший коммерческий разрыв — до монетизации: только 60,5% регистраций начинают KYC, и лишь 26,1% верифицированных пользователей покупают челлендж. Это точки роста без увеличения CAC — каждый процентный пункт улучшения конверсии KYC→покупка даёт ~86 дополнительных покупателей.

**Качество трейдера ≠ качество для бизнеса.**
Каналы `direct` и `community` приводят лучших трейдеров по метрикам прохождения, но именно они создают наибольшую экспозицию выплат. Оптимизация по конверсии в funded без учёта downstream-риска ведёт в неправильную сторону.

**Low-cost trials — инструмент привлечения, не метрика качества.**
Их показатели финансирования (8,1%) несопоставимы с основными продуктами. Включение trial-пользователей в общую статистику занижает воспринимаемое качество воронки.

**Поведение предсказывает исход заранее.**
Трейдеры с утверждёнными выплатами имеют в среднем 0,60% риска на сделку, win rate 50% и 0,50 нарушений. У трейдеров, провалившихся в фазе 1: 0,89% риска, win rate 42%, 1,15 нарушений. Эти признаки видны до стадии финансирования.

---

### EN — Key Takeaways

- The prop funnel breaks materially before monetization: KYC start and verified-to-purchase are the largest leak points.
- Downstream trader quality and business quality are not the same thing. Direct and community produce stronger traders but also larger payout liabilities.
- Low-cost trials are useful for acquisition, not for judging the health of the funded book.
- Behavior metrics are directionally predictive early enough to support product and risk interventions before funding decisions are made.